In [1]:
from pprint import pprint
import random
import pandas as pd
import numpy as np

import MOO_functions

In [2]:
# Damage scenario selection
ds_sel = 'DS2'

df = pd.read_excel(
    "DS_with_full_description.xlsx",
    sheet_name= ds_sel
)

# Repair time discretization (hours)
repair_time_interval = 0.25  # 15 minutes

# Build the reparations dictionary
time_reparation = {}

for _, row in df.iterrows():

    repair_time = float(row["fix time (hours)"])

    # Round up to the nearest interval
    repair_time = np.ceil(repair_time / repair_time_interval) * repair_time_interval

    time_reparation[str(row["Pipe ID"])] = {
        "t_r": repair_time
    }

print(f"{len(time_reparation)} repairs loaded.")
#time_reparation

106 repairs loaded.


In [3]:
# load the wdn as INP file WITH the broken pipes
input_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'

# Export the new INP file with the controls
output_inp="BBM-EPS_"+ds_sel+"_restoration.inp"

In [4]:
# Number of crews
n_crews = 3

# Pipe IDs from the reparations dictionary
pipe_ids = list(time_reparation.keys())

# Number of repairs
n_rep = len(pipe_ids)

In [5]:
indexes = [24,14,87,10,2,78,75,4,39,104,9,77,3,89,23,7,81,29,43,101,6,11,30,73,8,5,0,42,44,98,55,41,1,15,61,74,91,21,32,64,26,94,80,16,88,18,84,72,58,17,66,69,38,86,33,96,25,56,40,70,46,92,62,50,31,35,27,71,28,93,12,82,67,60,83,63,99,22,97,13,103,100,59,47,95,37,53,105,20,85,90,79,76,52,36,34,45,54,65,57,102,48,68,19,49,51]
print(len(indexes))

106


In [6]:
# # List of indexes randomly ordered corresponding to the pipe IDs to be repaired
# indexes = list(range(len(pipe_ids)))
# random.shuffle(indexes)
# print(indexes[:20])

In [7]:
# Import the travel time matrix from the Excel file
dmatrix_df = pd.read_parquet(
    "TravelTime_"+ds_sel+".parquet"
)

# Normalize IDs so the repair dictionary and matrix use the same keys
dmatrix_df.index = dmatrix_df.index.map(str)
dmatrix_df.columns = dmatrix_df.columns.map(str)

dmatrix_df.head()

,1951,3414,4988,5251,3404,6005,1252,2408,3094,3293,...,538,5488,5567,5677,5778,5959,6041,854,869,892
1951,0.25,0.25,0.25,0.75,0.25,0.75,0.50,0.25,0.25,0.25,...,0.25,0.75,0.75,0.75,0.75,0.75,0.75,0.50,0.50,0.50
3414,0.25,0.25,0.25,0.50,0.25,0.75,0.25,0.25,0.25,0.25,...,0.25,0.50,0.75,0.50,0.75,0.75,0.75,0.25,0.25,0.25
4988,0.25,0.25,0.25,0.50,0.25,0.75,0.25,0.25,0.25,0.25,...,0.25,0.75,0.75,0.50,0.75,0.75,0.75,0.25,0.25,0.25
5251,0.75,0.50,0.50,0.25,0.75,0.25,0.50,0.50,0.75,0.50,...,0.50,0.25,0.25,0.25,0.25,0.25,0.25,0.50,0.50,0.50
3404,0.25,0.25,0.25,0.75,0.25,0.75,0.25,0.25,0.25,0.25,...,0.25,0.75,0.75,0.75,0.75,0.75,0.75,0.25,0.25,0.25


In [8]:
schedule = MOO_functions.create_schedule(
    reparations=time_reparation, 
    dmatrix=dmatrix_df, 
    pipe_ids=pipe_ids, 
    indexes=indexes, 
    n_crews=n_crews
    )

schedule

,Order,Pipe,Crew,Travel,Repair,Start,Finish
0,1,2881,1,0.50,6.75,0.50,7.25
1,2,2508,2,0.50,3.50,0.50,4.00
2,3,4584,3,0.50,2.75,0.50,3.25
3,4,1186,3,0.50,4.50,3.75,8.25
4,5,4988,2,0.25,7.25,4.25,11.50
...,...,...,...,...,...,...,...
101,101,6041,1,0.75,2.75,134.50,137.25
102,103,2430,2,0.50,2.75,135.50,138.25
103,105,5691,3,0.50,3.25,137.75,141.00
104,104,4132,1,0.75,3.50,138.00,141.50


In [9]:
new_controls = MOO_functions.generate_controls(schedule=schedule)

new_controls

['; Pipe 2881',
 'LINK 2881_A CLOSED AT TIME 0.50',
 'LINK 2881_B CLOSED AT TIME 0.50',
 'LINK 2881 OPEN AT TIME 7.25',
 '; Pipe 2508',
 'LINK 2508_A CLOSED AT TIME 0.50',
 'LINK 2508_B CLOSED AT TIME 0.50',
 'LINK 2508 OPEN AT TIME 4.00',
 '; Pipe 4584',
 'LINK 4584_A CLOSED AT TIME 0.50',
 'LINK 4584_B CLOSED AT TIME 0.50',
 'LINK 4584 OPEN AT TIME 3.25',
 '; Pipe 1186',
 'LINK 1186_A CLOSED AT TIME 3.75',
 'LINK 1186_B CLOSED AT TIME 3.75',
 'LINK 1186 OPEN AT TIME 8.25',
 '; Pipe 4988',
 'LINK 4988_A CLOSED AT TIME 4.25',
 'LINK 4988_B CLOSED AT TIME 4.25',
 'LINK 4988 OPEN AT TIME 11.50',
 '; Pipe 3216',
 'LINK 3216_A CLOSED AT TIME 7.50',
 'LINK 3216_B CLOSED AT TIME 7.50',
 'LINK 3216 OPEN AT TIME 10.25',
 '; Pipe 3120',
 'LINK 3120_A CLOSED AT TIME 8.50',
 'LINK 3120_B CLOSED AT TIME 8.50',
 'LINK 3120 OPEN AT TIME 11.25',
 '; Pipe 3404',
 'LINK 3404_A CLOSED AT TIME 10.50',
 'LINK 3404_B CLOSED AT TIME 10.50',
 'LINK 3404 OPEN AT TIME 16.25',
 '; Pipe 2113',
 'LINK 2113_A CLOS

In [10]:
# Apply function to generate INP with controls
controls_inp = MOO_functions.write_inp_controls(
    input_inp=input_inp,
    output_inp=output_inp,
    schedule=schedule,
    new_controls=new_controls
)

print(f"Created: {controls_inp}")

Created: BBM-EPS_DS2_restoration.inp
